# Sultai mana-base explorer

Run all cells once. Then edit **Shock penalty** and **Town penalty** in the parameter cell and rerun the analysis cell below it. The expensive draw calculations stay cached in the tables.

Each penalty is charged per matching land in the base; Cavern has no composition penalty. The starting values are illustrative, not calibrated life costs. Modes have uniform weights within each physical spell profile.

Basic and nonbasic bounds are editable below. Exhaustive builds use the batch backend without grid-size cutoffs. The candidate count and capacity-grid size are printed before building. Town and Cavern are separate unrestricted rainbow sources; Cavern has no penalty.

The report scores the full pool once, selects only the top five rows, and reuses the score snapshot for both penalty crossings.


In [1]:
from pathlib import Path
import sys
from time import perf_counter
from math import prod
from html import escape

# Works when launched from the repository or one of its subdirectories.
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'mana_opportunities.py').exists())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from IPython.display import HTML, display
from tqdm.auto import tqdm
from mana_opportunities import CastingMode, ManaSource, Spell, SpellCost, VERGE_RULES, spell_components
from mana_search import ManaSpace, build_component_table


def show_table(headers, rows):
    header = ''.join(f'<th>{escape(str(h))}</th>' for h in headers)
    body = ''.join('<tr>' + ''.join(f'<td>{escape(str(v))}</td>' for v in row) + '</tr>' for row in rows)
    display(HTML(f'<table><thead><tr>{header}</tr></thead><tbody>{body}</tbody></table>'))


def source_name(source):
    if source.kind == 'verge':
        return f'verge:{source.colors}'
    suffix = f' (types: {source.types})' if source.kind == 'shock' and source.types else ''
    return f'{source.kind}:{source.colors}{suffix}' 


def swap_text(swap):
    if not swap.cards_replaced:
        return 'No replacement'
    out = ', '.join(f'{n} {source_name(s)}' for s, n in swap.out)
    incoming = ', '.join(f'{n} {source_name(s)}' for s, n in swap.incoming)
    return f'OUT {out}; IN {incoming}'


## Spell profiles and starting mana base

Edit these definitions here, then rerun this cell, build the search tables, and run the analysis. Counts in `MANA_BASE` describe the starting composition; the build cell allows 0?2 of each basic and caps shocks, Verges, Town, and Cavern at four each. The total land count is inferred from this composition. Mode weights default to uniform within each spell profile.

The shocks explicitly carry both basic land types. Verges, Town, and Cavern carry none. Verge secondary colors unlock when an enabling land type appears anywhere among lands seen, even if that land would not fit within the turn's land drops. This deliberately optimistic approximation avoids sequencing and battlefield search. Verges have no additional composition penalty.


In [2]:
SPELLS = (
    SpellCost("2UB", 4),  # Superior Spider-Man
    Spell(4, tuple(CastingMode(c) for c in ("UU", "BB", "4UU", "4BB", "2UUBB"))),  # Deceit
    Spell(4, tuple(CastingMode(c) for c in ("WW", "BB", "4WW", "4BB", "2WWBB"))),  # Emptiness
    Spell(4, tuple(CastingMode(c) for c in ("UU", "GG", "3UU", "3GG", "1UUGG"))),  # Wistfulness
    SpellCost("2G", 4),   # Formidable Speaker
    SpellCost("1B", 4),   # Bitter Triumph
    SpellCost("B", 4),    # Moonshadow
    SpellCost("7UU", 4),  # Sunderflock
    SpellCost("BGU", 2),  # Awaken the Honored Dead
)

MANA_BASE = {
    ManaSource("U", "basic"): 0,
    ManaSource("B", "basic"): 1,
    ManaSource("G", "basic"): 0,
    ManaSource("UB", "shock", types="UB"): 4,
    ManaSource("UG", "shock", types="UG"): 4,
    ManaSource("BG", "shock", types="BG"): 4,
    ManaSource("WUBRG", "town"): 4,
    ManaSource("WUBRG", "cavern"): 4,
    ManaSource("UB", "verge"): 3,
    ManaSource("BG", "verge"): 1,
    ManaSource("UG", "verge"): 1,
}


## Build search tables

Run this cell again only if you change the spell profile, bounds, or evaluation settings. The Sultai input uses an illustrative mana base with basics, typed shocks, separate Town/Cavern rainbow sources, and conditional Verges. Cavern restrictions are excluded.

Batch creation shares probability calculations across every base. Component bars report the expensive transforms; base bars report row assembly. The scalar backend remains available with `backend="scalar"` for exhaustive reference checks.


In [3]:

land_count = sum(MANA_BASE.values())
space = ManaSpace(
    {source: (0, 2 if source.kind == "basic" else 4) for source in MANA_BASE},
    land_count=land_count,
)
print(f'Exhaustive candidates: {space.candidate_count()}')
grid_cells = prod(hi + 1 for _, hi in space.bounds.values())
print(f'Capacity grid: {grid_cells:,} cells')
started = perf_counter()
table = build_component_table(
    space, SPELLS, baseline=MANA_BASE,
    progress=lambda candidates: tqdm(candidates, total=space.candidate_count(), desc='Base rows', unit='base'),
    component_progress=lambda costs: tqdm(costs, total=len(spell_components(SPELLS)[1]),
                                          desc='Components', unit='cost'),
)
print(f'{len(table.rows)} bases built in {perf_counter() - started:.2f}s')


Exhaustive candidates: 263274
Capacity grid: 10,546,875 cells


Components:   0%|          | 0/18 [00:00<?, ?cost/s]

Base rows:   0%|          | 0/263274 [00:00<?, ?base/s]

263274 bases built in 48.09s


## Parameters

Edit these two values, run this cell, then run the analysis cell. Zero disables that penalty. Changing these values does not require rebuilding the tables.


In [4]:
shock_penalty = 0.00687
town_penalty = 0.01  # Independent control; initially equal to shock_penalty.


## Analysis

Shows the five highest scores and concrete replacements from the original base. Sensitivity intervals describe each candidate set's current winner while varying one penalty and holding the other fixed. Endpoints include ties; crossing beyond an endpoint selects the displayed replacement. Fractions are retained in the underlying result objects.


In [5]:
penalties = dict(shock_penalty=shock_penalty, town_penalty=town_penalty)
started = perf_counter()
scored = table.score(**penalties)
ranked = scored.rank(limit=5)
winner = dict(zip(space.sources, ranked[0].counts))
display(HTML(f'<h3>Exhaustive: {len(table.rows):,} candidates</h3>'))
print(f'Shock penalty: {shock_penalty}; town penalty: {town_penalty}')
show_table(
    ['Rank', 'Raw opportunities', 'Penalty', 'Score', 'Gain from start', 'Replacements from start'],
    [(i, f'{float(r.raw_score):.6f}', f'{float(r.penalty):.6f}', f'{float(r.score):.6f}',
      f'{float(r.delta):+.6f}', swap_text(r.swap)) for i, r in enumerate(ranked[:5], 1)],
)
show_table(['Winning source', 'Count'], [(source_name(s), n) for s, n in winner.items() if n])
sensitivities = {}
rows = []
for parameter in ('shock_penalty', 'town_penalty'):
    sensitivity = scored.crossings(parameter, baseline=winner)
    sensitivities[parameter] = sensitivity
    interval = sensitivity.optimal_interval
    upper = 'unbounded' if interval.upper is None else f'{float(interval.upper):.8g}'
    rows.append((parameter, f'[{float(interval.lower):.8g}, {upper}]',
                 'none' if sensitivity.lower is None else swap_text(sensitivity.lower.swap),
                 'none' if sensitivity.upper is None else swap_text(sensitivity.upper.swap)))
show_table(['Varied penalty', 'Winner stays optimal', 'Below lower boundary', 'Above upper boundary'], rows)
analysis = dict(scored=scored, ranked=ranked, sensitivities=sensitivities)
print(f'Report computed in {perf_counter() - started:.2f}s')


Shock penalty: 0.00687; town penalty: 0.01


Rank,Raw opportunities,Penalty,Score,Gain from start,Replacements from start
1,3.280398,0.108700,3.171698,+0.002065,"OUT 2 shock:UG (types: UG), 1 verge:UB; IN 1 basic:B, 2 verge:UG"
2,3.273525,0.101830,3.171695,+0.002062,"OUT 3 shock:UG (types: UG), 2 verge:UB; IN 1 basic:B, 1 basic:U, 3 verge:UG"
3,3.280323,0.108700,3.171623,+0.001990,"OUT 1 shock:BG (types: BG), 1 shock:UG (types: UG), 1 verge:UB; IN 1 basic:B, 1 verge:BG, 1 verge:UG"
4,3.280203,0.108700,3.171503,+0.001871,"OUT 2 shock:UG (types: UG), 2 verge:UB; IN 1 basic:B, 1 verge:BG, 2 verge:UG"
5,3.280173,0.108700,3.171473,+0.001840,"OUT 2 shock:UG (types: UG), 1 verge:UB; IN 1 basic:B, 1 verge:BG, 1 verge:UG"


Winning source,Count
basic:B,2
cavern:WUBRG,4
shock:BG (types: BG),4
shock:UB (types: UB),4
shock:UG (types: UG),2
town:WUBRG,4
verge:BG,1
verge:UB,2
verge:UG,3


Varied penalty,Winner stays optimal,Below lower boundary,Above upper boundary
shock_penalty,"[0.006509117, 0.0068730019]","OUT 1 basic:B, 2 verge:UG; IN 1 shock:UG (types: UG), 1 verge:BG, 1 verge:UB","OUT 1 shock:UG (types: UG), 1 verge:UB; IN 1 basic:U, 1 verge:UG"
town_penalty,"[0, 0.02540952]",none,"OUT 1 town:WUBRG, 1 verge:UG; IN 1 shock:UG (types: UG), 1 verge:BG"


Report computed in 2.03s
